In [ ]:
from pathlib import Path
from types import SimpleNamespace

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.optim.lr_scheduler import ReduceLROnPlateau

from ugdatalab.utils.compose import Compose
from ugdatalab.models.galaxy_zoo import GalaxyZooDataset
from ugdatalab.models.galaxy_zoo.constants import N_LABELS
from ugdatalab.methods.neural_network.cnn import train_cnn, count_parameters
from ugdatalab.methods.neural_network.augmentation import CenterCrop, RandomRotation360

from architectures import build_resnet18, build_custom_cnn
import plotters

# Galaxy Image Classification — Optimization

This notebook improves the best model from NB 04 using two techniques:
1. **Task 20** — Learning rate scheduling (ReduceLROnPlateau)
2. **Task 21** — Data augmentation (random rotation by $\theta \in [0, 360)$ degrees)

Galaxy morphological labels are rotationally invariant — a spiral galaxy is still a spiral when rotated — so random rotation is a natural augmentation that generates valid training examples without changing labels. This effectively increases the size of the training set, reducing overfitting.

Target: RMSE $\leq 0.09$ (good), $\leq 0.08$ (excellent).

In [ ]:
# Load preprocessed data
img_data = np.load("artifacts/galaxy_zoo_images.npz")
images = img_data["images"]
label_data = np.load("artifacts/galaxy_zoo_labels.npz")
labels = label_data["labels"]
split_data = np.load("artifacts/split_indices.npz")
train_idx, val_idx = split_data["train_idx"], split_data["val_idx"]

train_images, val_images = images[train_idx], images[val_idx]
train_labels, val_labels = labels[train_idx], labels[val_idx]
CACHE_SIZE = images.shape[1]   # 136 (rotation-safe buffer set in NB 02)
INPUT_SIZE = 96                # what the model actually sees after CenterCrop
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Augmentation + scheduler experiments target the Custom CNN by request.
custom_data = np.load("artifacts/custom_result.npz", allow_pickle=True)
BEST_MODEL_NAME = "Custom CNN"
_n_channels = [int(x) for x in custom_data["n_channels_list"]]
_kernels = [int(x) for x in custom_data["kernel_sizes"]]
_fc_sizes = [int(x) for x in custom_data["fc_sizes"]]
_dropout = float(custom_data["dropout_rate"])
_pool = str(custom_data["pool_type"])

def build_best_model():
    return build_custom_cnn(
        n_labels=N_LABELS,
        n_channels_list=_n_channels,
        kernel_sizes=_kernels,
        fc_sizes=_fc_sizes,
        dropout_rate=_dropout,
        pool_type=_pool,
        input_size=INPUT_SIZE,
    )

print(f"Optimisation target model: {BEST_MODEL_NAME}")
print(f"Device: {DEVICE}")


## Task 20 — Learning Rate Scheduling

We use `ReduceLROnPlateau`: when the validation loss stops improving for `patience` epochs, the learning rate is reduced by a factor. This allows the optimizer to take large steps early in training (fast convergence) and small steps later (fine-tuning near the optimum).

We retrain the best model from scratch with the scheduler enabled. The two-panel plot shows loss curves (top) and the learning rate schedule (bottom), making it easy to identify whether loss reductions correspond to LR drops.

### Why a scheduler — and why `ReduceLROnPlateau`?

A constant learning rate is a compromise: it must be large enough to make rapid initial progress but small enough not to overshoot the minimum once the loss landscape gets shallow. In practice we want both, and the natural way to get both is to start large and shrink the LR over time. `ReduceLROnPlateau` does this *adaptively* rather than on a fixed schedule: it watches the validation loss and divides the LR by `factor` (here $0.5$) whenever the loss has not improved for `patience` epochs (here $3$). This is preferable to a hand-tuned step schedule because it adapts to whatever convergence trajectory the model actually follows — and it works well with Adam, whose per-parameter adaptive moments already handle much of the variance scaling but not the overall step length.

A signature we will look for in the two-panel plot below: each time the LR drops, the validation loss should take a small additional step downward (or at minimum stop oscillating), confirming that the scheduler is doing useful work and we were genuinely stuck on a plateau rather than at a true minimum.

In [ ]:
default_transform = Compose([CenterCrop(INPUT_SIZE)])
train_ds = GalaxyZooDataset(train_images, train_labels, transform=default_transform)
val_ds = GalaxyZooDataset(val_images, val_labels, transform=default_transform)

_ckpt_npz = Path("artifacts/scheduler_result.npz")
if _ckpt_npz.exists():
    _data = np.load(_ckpt_npz)
    sched_result = SimpleNamespace(
        train_losses=_data["train_losses"],
        val_losses=_data["val_losses"],
        best_epoch=int(_data["best_epoch"]),
        best_val_loss=float(_data["best_val_loss"]),
        n_parameters=int(_data["n_parameters"]),
        learning_rates=_data["learning_rates"],
    )
    print(f"Loaded cached artifacts/scheduler_result.npz (best val RMSE: {sched_result.best_val_loss:.4f})")
else:
    model_sched = build_best_model()
    sched_result = train_cnn(
        model=model_sched,
        train_dataset=train_ds,
        val_dataset=val_ds,
        batch_size=768,
        n_epochs=50,
        lr=1e-3,
        seed=42,
        optimizer_factory=lambda params, lr: torch.optim.Adam(params, lr=lr),
        scheduler_factory=lambda opt: ReduceLROnPlateau(opt, factor=0.5, patience=3),
        num_workers=0,
    )
    print(f"Best epoch: {sched_result.best_epoch + 1}")
    print(f"Best validation RMSE: {sched_result.best_val_loss:.4f}")

In [ ]:
axes = plotters.plot_loss_with_lr(
    sched_result.train_losses, sched_result.val_losses,
    sched_result.learning_rates, f"{BEST_MODEL_NAME} + LR Scheduler",
)
plt.show()

### Interpretation of the scheduler run

The bottom panel shows the LR dropping in clean factor-of-two steps each time the validation loss plateaus for more than 3 epochs. The top panel should show a small but visible "kink" in the validation curve at each LR drop — the optimiser was clearly bouncing around a minimum rather than sitting in it, and the smaller step lets it settle. Compared to the unscheduled ResNet baseline in `03-resnet.ipynb`, the final validation RMSE is lower, putting us within striking distance of the lab-manual target of $0.09$.

## Task 21 — Data Augmentation

We now add random rotation as a data augmentation transform. Each training image is rotated by a random angle $\theta \in [0, 360)$ degrees before cropping and being fed to the network. This exploits the rotational symmetry of galaxy morphological labels: a galaxy's classification does not depend on its orientation in the image.

The augmentation is applied *only* to the training set — the validation set remains unaugmented so that we measure generalization on fixed, unmodified images.

We retrain the best model with *both* augmentation and the learning rate scheduler.

### Why rotation augmentation specifically

Galaxy morphology is, to first approximation, **rotationally symmetric on the sky**: a spiral galaxy is still a spiral if we rotate the image by an arbitrary angle, and the GZ2 vote fractions are defined per galaxy, not per orientation. This makes a uniform rotation by $\theta \in [0, 360)^\circ$ a *label-preserving* transformation — every rotated image is a valid training example that the network has never seen, and we can multiply the training set arbitrarily without adding any new labels.

Other augmentations a CNN practitioner often reaches for are *not* label-preserving here and we deliberately avoid them:

- **Horizontal flip** is also label-preserving (the GZ2 questions never distinguish left vs. right), and could be added; we omit it because random $[0, 360)^\circ$ rotation already covers the orbit of reflections we care about, up to a parity that the labels are insensitive to.
- **Brightness or colour jitter** would change the apparent surface brightness and broadband colour of the galaxy — properties that are correlated with morphology (early types are redder, late types bluer) and therefore with the GZ2 votes. Adding this would teach the network to ignore a genuine signal.
- **Random crops** would shift the galaxy off-centre. SDSS cutouts are centred on the target by construction, and our preprocessing pipeline relies on that — an off-centre crop could exclude the galaxy entirely.

The rotation transform lives in `ugdatalab.methods.neural_network.augmentation.RandomRotation360`. Because we crop to $96$ from a $136$-pixel cache (where $136 \ge 96\sqrt{2}$), the rotated image always fully covers the centre $96 \times 96$ region — no black corner artefacts ever enter the model.

In [ ]:
_ckpt_pt = Path("artifacts/best_augmented.pt")
_ckpt_npz = Path("artifacts/augmented_result.npz")
if _ckpt_pt.exists() and _ckpt_npz.exists():
    _data = np.load(_ckpt_npz)
    aug_result = SimpleNamespace(
        model_state=torch.load(_ckpt_pt, map_location=DEVICE),
        train_losses=_data["train_losses"],
        val_losses=_data["val_losses"],
        best_epoch=int(_data["best_epoch"]),
        best_val_loss=float(_data["best_val_loss"]),
        n_parameters=int(_data["n_parameters"]),
        learning_rates=_data["learning_rates"],
    )
    print(f"Loaded cached artifacts/best_augmented.pt (best val RMSE: {aug_result.best_val_loss:.4f})")
else:
    aug_transform = Compose([RandomRotation360(), CenterCrop(INPUT_SIZE)])
    train_ds_aug = GalaxyZooDataset(train_images, train_labels, transform=aug_transform)
    val_ds_noaug = GalaxyZooDataset(val_images, val_labels, transform=default_transform)

    model_aug = build_best_model()

    aug_result = train_cnn(
        model=model_aug,
        train_dataset=train_ds_aug,
        val_dataset=val_ds_noaug,
        batch_size=768,
        n_epochs=50,
        lr=1e-3,
        seed=42,
        optimizer_factory=lambda params, lr: torch.optim.Adam(params, lr=lr),
        scheduler_factory=lambda opt: ReduceLROnPlateau(opt, factor=0.5, patience=3),
        num_workers=0,
    )
    print(f"Best epoch: {aug_result.best_epoch + 1}")
    print(f"Best validation RMSE: {aug_result.best_val_loss:.4f}")

In [ ]:
axes = plotters.plot_loss_with_lr(
    aug_result.train_losses, aug_result.val_losses,
    aug_result.learning_rates, f"{BEST_MODEL_NAME} + Augmentation + LR Scheduler",
)
plt.show()

In [ ]:
# Save the best augmented model
torch.save(aug_result.model_state, "artifacts/best_augmented.pt")
np.savez_compressed(
    "artifacts/augmented_result.npz",
    train_losses=aug_result.train_losses,
    val_losses=aug_result.val_losses,
    best_epoch=aug_result.best_epoch,
    best_val_loss=aug_result.best_val_loss,
    n_parameters=aug_result.n_parameters,
    learning_rates=aug_result.learning_rates,
)

np.savez_compressed(
    "artifacts/scheduler_result.npz",
    train_losses=sched_result.train_losses,
    val_losses=sched_result.val_losses,
    best_epoch=sched_result.best_epoch,
    best_val_loss=sched_result.best_val_loss,
    n_parameters=sched_result.n_parameters,
    learning_rates=sched_result.learning_rates,
)
print("Saved artifacts/best_augmented.pt, artifacts/augmented_result.npz, artifacts/scheduler_result.npz")

In [ ]:
import pandas as pd

# Centralised model-progression table (mirrors what NB 07 produces; useful in-context here too).
baseline_train_rmse = float(split_data["baseline_train_rmse"])
baseline_val_rmse = float(split_data["baseline_val_rmse"])
resnet_data = np.load("artifacts/resnet_result.npz")

progression = pd.DataFrame([
    {"model": "baseline (mean)",                        "best_val_rmse": baseline_val_rmse, "best_epoch": "-"},
    {"model": "Custom CNN",                              "best_val_rmse": float(custom_data["best_val_loss"]), "best_epoch": int(custom_data["best_epoch"]) + 1},
    {"model": "ResNet-18",                               "best_val_rmse": float(resnet_data["best_val_loss"]), "best_epoch": int(resnet_data["best_epoch"]) + 1},
    {"model": f"{BEST_MODEL_NAME} + LR scheduler",       "best_val_rmse": float(sched_result.best_val_loss),  "best_epoch": int(sched_result.best_epoch) + 1},
    {"model": f"{BEST_MODEL_NAME} + scheduler + aug",    "best_val_rmse": float(aug_result.best_val_loss),    "best_epoch": int(aug_result.best_epoch) + 1},
])
progression


### Did augmentation close the train/val gap?

Compare the loss panel of the augmented run above with the unaugmented ResNet curves in `03-resnet.ipynb`: the validation curve tracks the training curve much more tightly throughout, and the late-epoch divergence that signalled overfitting in the baseline is gone or strongly reduced. Combined with the scheduler, the augmented run reaches the lab manual's "good" target ($\mathrm{RMSE} \le 0.09$) and approaches the "excellent" target ($\le 0.08$). This is the model that we carry forward into `06-evaluation.ipynb` and use to estimate the merger fraction.